In [6]:
import os
from dotenv import load_dotenv
import json
from mistralai import Mistral, DocumentURLChunk
import tqdm
load_dotenv()

True

In [8]:
def OCR_On_Folder_Content(raw_data_path:str, intermediate_data_path:str):
    """
    Processes all files in a specified folder for OCR (Optical Character Recognition) and
    saves both JSON and text content outputs into an intermediate directory.

    :param raw_data_path: Path to the directory containing raw data files to process.
    :type raw_data_path: str
    :param intermediate_data_path: Path to the directory where processed JSON and text
        files will be saved.
    :type intermediate_data_path: str
    :return: None
    """
    client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
    for filename in tqdm.tqdm(os.listdir(raw_data_path)):
        # Read file and upload
        try:
            with open(os.path.join(raw_data_path, filename), "rb") as file_content:
                uploaded_file = client.files.upload(
                    file={
                        "file_name": os.path.join(raw_data_path, filename),
                        "content": file_content,
                    },
                    purpose="ocr",
                )
        except Exception as e:
            raise RuntimeError(f"Error uploading file: {e}")

        # Get signed URL
        try:
            signed_url = client.files.get_signed_url(file_id=uploaded_file.id)
        except Exception as e:
            raise RuntimeError(f"Error generating signed URL: {e}")

        # Process OCR
        try:
            pdf_response = client.ocr.process(
                document=DocumentURLChunk(document_url=signed_url.url),
                model="mistral-ocr-latest",
                include_image_base64=False,
            )
            response_dict = json.loads(pdf_response.model_dump_json())
        except Exception as e:
            raise RuntimeError(f"Error processing OCR: {e}")

        # Save processed OCR results
        base_filename, _ = os.path.splitext(filename)
        new_filename = f"{base_filename}.json"
        output_path = os.path.join(intermediate_data_path, new_filename)

        # Handle file saving
        try:
            if os.path.exists(output_path):
                os.remove(output_path)
            with open(output_path, "w") as f:
                json.dump(response_dict, f, indent=4, ensure_ascii=False)
        except Exception as e:
            raise RuntimeError(f"Error saving OCR result: {e}")

        # Combine markdown content
        base_filename, _ = os.path.splitext(filename)
        new_filename = f"{base_filename}.txt"
        output_path = os.path.join(intermediate_data_path, new_filename)
        chunk_content_string = "\n".join(chunk["markdown"] for chunk in response_dict["pages"])
        try:
            # 3. If an existing file already exists at that path, remove it
            if os.path.exists(output_path):  # :contentReference[oaicite:1]{index=1}
                os.remove(output_path)

            # 4. Open a brand-new .txt file (or overwrite if it somehow still exists)
            #    Use UTF-8 encoding to ensure any non-ASCII characters are handled correctly :contentReference[oaicite:2]{index=2}
            with open(output_path, "w", encoding="utf-8") as f:
                # 5. Write the combined markdown string into the .txt file
                f.write(chunk_content_string)  # :contentReference[oaicite:3]{index=3}

        except Exception as e:
            raise RuntimeError(f"Error saving OCR result: {e}")


raw_folder = "../Data/RAG/Illinois/Raw"
intermediate_folder = "../Data/RAG/Illinois/Intermediate"
OCR_On_Folder_Content(raw_folder, intermediate_folder)

100%|██████████| 15/15 [01:27<00:00,  5.81s/it]
